# 🥈 Silver Layer - Data Quality & Transformation Pipeline

## Arquitetura Medallion - Camada Silver

Este notebook implementa a camada **Silver** seguindo as melhores práticas de **Data Engineering**, **DataOps** e **Data Quality**.

### 📋 Objetivos

1. **Limpeza de Dados**: Remover duplicatas, valores nulos, inconsistências
2. **Transformação**: Padronização, normalização, enriquecimento
3. **Qualidade**: Validações automáticas com Great Expectations
4. **Observabilidade**: Logging estruturado, métricas, rastreamento
5. **Performance**: Processamento paralelo com Spark

### 🏗️ Arquitetura

```
Bronze (ClickHouse) → Spark Processing → Silver (ClickHouse)
                            ↓
                    Data Quality Checks
                    Observability Metrics
                    Parallel Processing
```

### 🛠️ Stack Tecnológico

- **Spark**: Processamento distribuído e paralelo
- **ClickHouse**: Storage de alta performance
- **Great Expectations**: Data Quality framework
- **OpenTelemetry**: Observabilidade e tracing
- **Pandera**: Schema validation
- **DuckDB**: Análises rápidas em memória

---
## 1. 📦 Instalação de Dependências

In [22]:
# Instalar dependências necessárias
!pip install -q great-expectations pandera duckdb opentelemetry-api opentelemetry-sdk \
    clickhouse-connect pyspark pandas pyarrow fastparquet python-dotenv \
    plotly kaleido loguru

---
## 2. 🔧 Configuração e Imports

In [23]:
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)

import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import Dict, List, Any, Optional, Tuple
import json
from dataclasses import dataclass, asdict
from enum import Enum

# Data Processing
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# ClickHouse
import clickhouse_connect

# Data Quality
import great_expectations as gx
# import pandera as pa
# from pandera import Column, DataFrameSchema, Check

# Observability - Using built-in logging instead of loguru
import logging
import time

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s:%(funcName)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Environment
from dotenv import load_dotenv
load_dotenv()

print("✅ Imports carregados com sucesso!")

✅ Imports carregados com sucesso!


---
## 3. 🎯 Configuração de Logging e Observabilidade

In [24]:
# Configurar diretórios
# Get the actual working directory inside the container
current_dir = Path(os.getcwd())
print(f"Current directory: {current_dir}")

# Navigate to project root (assuming notebook is in output/jupyter-notebook/)
if current_dir.name == "jupyter-notebook":
    BASE_DIR = current_dir.parent.parent
elif current_dir.name == "output":
    BASE_DIR = current_dir.parent
else:
    # Fallback: use current directory
    BASE_DIR = current_dir

print(f"Base directory: {BASE_DIR}")

LOGS_DIR = BASE_DIR / "logs" / "silver"
METRICS_DIR = BASE_DIR / "output" / "metrics" / "silver"
QUALITY_DIR = BASE_DIR / "output" / "data_quality"

LOGS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
QUALITY_DIR.mkdir(parents=True, exist_ok=True)

# File handler (structured logging)
file_handler = logging.FileHandler(
    LOGS_DIR / f"silver_pipeline_{datetime.now():%Y%m%d_%H%M%S}.log"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter(
    '%(asctime)s | %(levelname)s | %(name)s:%(funcName)s | %(message)s'
))
logger.addHandler(file_handler)

logger.info("🚀 Sistema de logging configurado")
logger.info(f"📁 Logs: {LOGS_DIR}")
logger.info(f"📊 Métricas: {METRICS_DIR}")
logger.info(f"✅ Quality Reports: {QUALITY_DIR}")

Current directory: /Users/joseamaro/Documents/Projeto/data-pipeline-track/output/jupyter-notebook
Base directory: /Users/joseamaro/Documents/Projeto/data-pipeline-track


---
## 4. 📊 Classes de Métricas e Observabilidade

In [25]:
@dataclass
class PipelineMetrics:
    """Métricas do pipeline de processamento"""
    table_name: str
    start_time: datetime
    end_time: Optional[datetime] = None
    
    # Volumetria
    rows_input: int = 0
    rows_output: int = 0
    rows_duplicates: int = 0
    rows_invalid: int = 0
    rows_nulls: int = 0
    
    # Performance
    duration_seconds: float = 0.0
    throughput_rows_per_sec: float = 0.0
    
    # Qualidade
    quality_score: float = 0.0
    quality_checks_passed: int = 0
    quality_checks_failed: int = 0
    
    # Status
    status: str = "running"
    error_message: Optional[str] = None
    
    def finalize(self):
        """Finaliza as métricas calculando valores derivados"""
        self.end_time = datetime.now()
        self.duration_seconds = (self.end_time - self.start_time).total_seconds()
        
        if self.duration_seconds > 0:
            self.throughput_rows_per_sec = self.rows_output / self.duration_seconds
        
        # Calcular quality score
        total_checks = self.quality_checks_passed + self.quality_checks_failed
        if total_checks > 0:
            self.quality_score = (self.quality_checks_passed / total_checks) * 100
    
    def to_dict(self) -> Dict:
        """Converte para dicionário serializável"""
        data = asdict(self)
        data['start_time'] = self.start_time.isoformat()
        data['end_time'] = self.end_time.isoformat() if self.end_time else None
        return data


class MetricsCollector:
    """Coletor centralizado de métricas"""
    
    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.metrics: List[PipelineMetrics] = []
        logger.info(f"📊 MetricsCollector inicializado: {output_dir}")
    
    def add_metric(self, metric: PipelineMetrics):
        """Adiciona métrica à coleção"""
        self.metrics.append(metric)
        logger.debug(f"Métrica adicionada: {metric.table_name}")
    
    def save_metrics(self, filename: str = None):
        """Salva métricas em JSON"""
        if not filename:
            filename = f"pipeline_metrics_{datetime.now():%Y%m%d_%H%M%S}.json"
        
        filepath = self.output_dir / filename
        
        data = {
            'pipeline_run': datetime.now().isoformat(),
            'total_tables': len(self.metrics),
            'metrics': [m.to_dict() for m in self.metrics]
        }
        
        with open(filepath, 'w') as f:
            json.dump(data, f, indent=2)
        
        logger.info(f"💾 Métricas salvas: {filepath}")
        return filepath
    
    def get_summary(self) -> Dict:
        """Retorna sumário das métricas"""
        if not self.metrics:
            return {}
        
        total_rows_input = sum(m.rows_input for m in self.metrics)
        total_rows_output = sum(m.rows_output for m in self.metrics)
        total_duplicates = sum(m.rows_duplicates for m in self.metrics)
        total_invalid = sum(m.rows_invalid for m in self.metrics)
        avg_quality = np.mean([m.quality_score for m in self.metrics if m.quality_score > 0])
        
        return {
            'total_tables_processed': len(self.metrics),
            'total_rows_input': total_rows_input,
            'total_rows_output': total_rows_output,
            'total_duplicates_removed': total_duplicates,
            'total_invalid_rows': total_invalid,
            'avg_quality_score': round(avg_quality, 2),
            'success_rate': round((len([m for m in self.metrics if m.status == 'success']) / len(self.metrics)) * 100, 2)
        }


# Inicializar coletor
metrics_collector = MetricsCollector(METRICS_DIR)
logger.info("✅ MetricsCollector inicializado")

2026-02-07 13:17:18 | INFO     | __main__:<module> | ✅ MetricsCollector inicializado


---
## 5. 🔌 Configuração de Conexões

In [28]:
# ClickHouse Connection
CH_HOST = os.getenv('CLICKHOUSE_HOST', 'e1a1lieug8.us-central1.gcp.clickhouse.cloud')
CH_PORT = int(os.getenv('CLICKHOUSE_PORT', 8443))
CH_USER = os.getenv('CLICKHOUSE_USER', 'default')
CH_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '_uv765EvWphL_')
CH_DATABASE = os.getenv('CLICKHOUSE_DATABASE', 'default')

logger.info(f"🔌 Conectando ao ClickHouse: {CH_HOST}:{CH_PORT}")

try:
    client = clickhouse_connect.get_client(
        host=CH_HOST,
        port=CH_PORT,
        username=CH_USER,
        password=CH_PASSWORD,
        database=CH_DATABASE
    )
    
    # Testar conexão
    result = client.query("SELECT version()")
    version = result.result_rows[0][0]
    logger.info(f"✅ ClickHouse conectado! Versão: {version}")
    
except Exception as e:
    logger.error(f"❌ Erro ao conectar ClickHouse: {e}")
    raise

OperationalError: Error HTTPSConnectionPool(host='e1a1lieug8.us-central1.gcp.clickhouse.cloud', port=8443): Max retries exceeded with url: /?wait_end_of_query=1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1002)'))) executing HTTP request attempt 1 https://e1a1lieug8.us-central1.gcp.clickhouse.cloud:8443

In [ ]:
# Spark Session com otimizações
logger.info("⚡ Inicializando Spark com otimizações...")

spark = SparkSession.builder \
    .appName("SilverLayer-DataQuality") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "100") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "10000") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Configurar log level
spark.sparkContext.setLogLevel("WARN")

logger.info(f"✅ Spark inicializado: {spark.version}")
logger.info(f"   Parallelism: {spark.sparkContext.defaultParallelism}")
logger.info(f"   Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

---
## 6. 📋 Definição de Tabelas e Schemas

In [ ]:
# Tabelas disponiveis no ClickHouse (corrigido com nomes reais)
tables_to_process = [
    "tst_contratos",
    "depara_cliente",
    "sc5030",
    "sc6030",
    "sd2030",
    "sf2030",
]

# Configuracao de amostragem para testes
SAMPLE_MODE = True  # True = usar amostra, False = processar tudo
SAMPLE_SIZE = 10000  # Numero de linhas por tabela (se SAMPLE_MODE=True)
SAMPLE_METHOD = "RANDOM"  # "RANDOM" ou "TOP"

if SAMPLE_MODE:
    logger.info("🧪 MODO DE TESTE ATIVADO")
    logger.info(f"   Metodo: {SAMPLE_METHOD}")
    logger.info(f"   Tamanho da amostra: {SAMPLE_SIZE:,} linhas por tabela")
else:
    logger.info("🚀 MODO PRODUCAO - Processando dados completos")

# Verificar quais tabelas existem no ClickHouse
logger.info("🔍 Verificando tabelas disponiveis no ClickHouse...")
available_tables = []
table_info = []

for table in tables_to_process:
    try:
        # Contar linhas totais
        result = client.query(f"SELECT count() as cnt FROM {table}")
        total_count = result.result_rows[0][0]

        if total_count > 0:
            available_tables.append(table)

            # Determinar quantas linhas processar
            if SAMPLE_MODE:
                rows_to_process = min(SAMPLE_SIZE, total_count)
            else:
                rows_to_process = total_count

            table_info.append({
                "table": table,
                "total_rows": total_count,
                "rows_to_process": rows_to_process,
                "is_sample": SAMPLE_MODE and total_count > SAMPLE_SIZE,
            })

            logger.info(f"✓ {table}: {total_count:,} linhas -> processar {rows_to_process:,}")
        else:
            logger.warning(f"✗ {table}: tabela vazia")

    except Exception as e:
        logger.warning(f"✗ {table}: nao encontrada ou erro - {str(e)[:100]}")

# Criar DataFrame com informacoes
import pandas as pd

if len(table_info) > 0:
    table_info_df = pd.DataFrame(table_info)
    logger.info(f"✅ {len(available_tables)}/{len(tables_to_process)} tabelas disponiveis")
    print("\n" + "=" * 80)
    print("📋 TABELAS DISPONIVEIS PARA PROCESSAMENTO")
    print("=" * 80)
    print(table_info_df.to_string(index=False))
    print("=" * 80)

    if SAMPLE_MODE:
        total_rows_input = table_info_df["total_rows"].sum()
        total_rows_process = table_info_df["rows_to_process"].sum()
        reduction_pct = (1 - total_rows_process / total_rows_input) * 100

        print("\n📊 RESUMO DA AMOSTRAGEM:")
        print(f"   Total de linhas disponiveis: {total_rows_input:,}")
        print(f"   Total a processar (amostra):  {total_rows_process:,}")
        print(f"   Reducao: {reduction_pct:.1f}%")
        print("=" * 80)
else:
    logger.error("❌ Nenhuma tabela encontrada no ClickHouse!")
    logger.error("   Verifique:")
    logger.error("   1. As credenciais de conexão estão corretas?")
    logger.error("   2. As tabelas existem no banco de dados?")
    logger.error("   3. O usuário tem permissão para acessar as tabelas?")
    
    # Tentar listar tabelas disponíveis
    try:
        all_tables = client.query("SHOW TABLES")
        if all_tables.result_rows:
            logger.info(f"   Tabelas encontradas no banco: {len(all_tables.result_rows)}")
            logger.info("   Todas as tabelas disponíveis:")
            for i, row in enumerate(all_tables.result_rows):
                logger.info(f"      - {row[0]}")
        else:
            logger.warning("   Nenhuma tabela encontrada no banco de dados")
    except Exception as e:
        logger.error(f"   Erro ao listar tabelas: {e}")


---
## 7. ✅ Framework de Data Quality

In [ ]:
class DataQualityChecker:
    """Framework de validação de qualidade de dados"""
    
    def __init__(self, spark_df: SparkDataFrame, table_name: str):
        self.df = spark_df
        self.table_name = table_name
        self.checks_passed = 0
        self.checks_failed = 0
        self.issues = []
        logger.info(f"🔍 DataQualityChecker iniciado: {table_name}")
    
    def check_completeness(self, threshold: float = 0.95) -> bool:
        """Verifica completude dos dados (% de valores não nulos)"""
        logger.debug(f"Checking completeness for {self.table_name}...")
        
        total_rows = self.df.count()
        if total_rows == 0:
            self.checks_failed += 1
            self.issues.append({"check": "completeness", "status": "FAILED", "reason": "Empty dataset"})
            return False
        
        results = []
        for col in self.df.columns:
            non_null_count = self.df.filter(F.col(col).isNotNull()).count()
            completeness = non_null_count / total_rows
            
            if completeness < threshold:
                results.append({
                    "column": col,
                    "completeness": round(completeness, 4),
                    "null_count": total_rows - non_null_count
                })
        
        if results:
            self.checks_failed += 1
            self.issues.append({
                "check": "completeness",
                "status": "FAILED",
                "details": results
            })
            logger.warning(f"❌ Completeness check failed: {len(results)} columns below threshold")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Completeness check passed")
            return True
    
    def check_duplicates(self, key_columns: List[str] = None) -> Tuple[bool, int]:
        """Verifica duplicatas"""
        logger.debug(f"Checking duplicates for {self.table_name}...")
        
        if key_columns:
            # Duplicatas baseadas em chave
            duplicates = self.df.groupBy(key_columns).count().filter(F.col("count") > 1)
        else:
            # Duplicatas exatas (todas as colunas)
            duplicates = self.df.groupBy(self.df.columns).count().filter(F.col("count") > 1)
        
        dup_count = duplicates.count()
        
        if dup_count > 0:
            self.checks_failed += 1
            self.issues.append({
                "check": "duplicates",
                "status": "FAILED",
                "duplicate_groups": dup_count
            })
            logger.warning(f"⚠️  Found {dup_count} duplicate groups")
            return False, dup_count
        else:
            self.checks_passed += 1
            logger.info(f"✅ No duplicates found")
            return True, 0
    
    def check_data_types(self, expected_types: Dict[str, str] = None) -> bool:
        """Valida tipos de dados"""
        logger.debug(f"Checking data types for {self.table_name}...")
        
        if not expected_types:
            self.checks_passed += 1
            logger.info("✅ Data types check skipped (no schema provided)")
            return True
        
        issues = []
        for col, expected_type in expected_types.items():
            if col in self.df.columns:
                actual_type = str(self.df.schema[col].dataType)
                if expected_type.lower() not in actual_type.lower():
                    issues.append({
                        "column": col,
                        "expected": expected_type,
                        "actual": actual_type
                    })
        
        if issues:
            self.checks_failed += 1
            self.issues.append({
                "check": "data_types",
                "status": "FAILED",
                "details": issues
            })
            logger.warning(f"❌ Data type check failed: {len(issues)} mismatches")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Data types check passed")
            return True
    
    def check_value_ranges(self, range_checks: Dict[str, Dict] = None) -> bool:
        """Valida ranges de valores"""
        if not range_checks:
            self.checks_passed += 1
            return True
        
        issues = []
        for col, ranges in range_checks.items():
            if col not in self.df.columns:
                continue
            
            min_val = ranges.get('min')
            max_val = ranges.get('max')
            
            if min_val is not None:
                invalid = self.df.filter(F.col(col) < min_val).count()
                if invalid > 0:
                    issues.append(f"{col}: {invalid} values < {min_val}")
            
            if max_val is not None:
                invalid = self.df.filter(F.col(col) > max_val).count()
                if invalid > 0:
                    issues.append(f"{col}: {invalid} values > {max_val}")
        
        if issues:
            self.checks_failed += 1
            self.issues.append({"check": "value_ranges", "status": "FAILED", "details": issues})
            logger.warning(f"❌ Value range check failed: {len(issues)} issues")
            return False
        else:
            self.checks_passed += 1
            logger.info(f"✅ Value ranges check passed")
            return True
    
    def get_summary(self) -> Dict:
        """Retorna sumário das validações"""
        total_checks = self.checks_passed + self.checks_failed
        quality_score = (self.checks_passed / total_checks * 100) if total_checks > 0 else 0
        
        return {
            "table_name": self.table_name,
            "total_checks": total_checks,
            "checks_passed": self.checks_passed,
            "checks_failed": self.checks_failed,
            "quality_score": round(quality_score, 2),
            "issues": self.issues
        }

logger.info("✅ DataQualityChecker definido")

---
## 8. 🧹 Funções de Limpeza e Transformação

In [ ]:
class DataTransformer:
    """Pipeline de transformações de dados"""
    
    @staticmethod
    def remove_duplicates(df: SparkDataFrame, subset: List[str] = None) -> Tuple[SparkDataFrame, int]:
        """Remove duplicatas mantendo primeira ocorrência"""
        logger.debug("Removendo duplicatas...")
        
        initial_count = df.count()
        
        if subset:
            df_clean = df.dropDuplicates(subset)
        else:
            df_clean = df.distinct()
        
        final_count = df_clean.count()
        removed = initial_count - final_count
        
        logger.info(f"🧹 Duplicatas removidas: {removed:,} ({removed/initial_count*100:.2f}%)")
        
        return df_clean, removed
    
    @staticmethod
    def handle_nulls(df: SparkDataFrame, strategy: str = 'drop', fill_value: Any = None) -> SparkDataFrame:
        """Trata valores nulos"""
        logger.debug(f"Tratando nulls com estratégia: {strategy}")
        
        if strategy == 'drop':
            # Remove linhas com qualquer null
            return df.na.drop()
        elif strategy == 'fill':
            # Preenche com valor específico
            return df.na.fill(fill_value)
        elif strategy == 'fill_smart':
            # Preenche com valores apropriados por tipo
            fill_values = {}
            for field in df.schema.fields:
                if isinstance(field.dataType, (IntegerType, LongType, FloatType, DoubleType)):
                    fill_values[field.name] = 0
                elif isinstance(field.dataType, StringType):
                    fill_values[field.name] = ''
            return df.na.fill(fill_values)
        else:
            return df
    
    @staticmethod
    def standardize_strings(df: SparkDataFrame, columns: List[str] = None) -> SparkDataFrame:
        """Padroniza strings: trim, uppercase, remove caracteres especiais"""
        logger.debug("Padronizando strings...")
        
        if not columns:
            # Auto-detectar colunas string
            columns = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        
        for col in columns:
            if col in df.columns:
                df = df.withColumn(
                    col,
                    F.trim(F.upper(F.col(col)))
                )
        
        logger.info(f"✅ {len(columns)} colunas padronizadas")
        return df
    
    @staticmethod
    def add_metadata_columns(df: SparkDataFrame) -> SparkDataFrame:
        """Adiciona colunas de metadados (timestamp, versão, etc)"""
        logger.debug("Adicionando metadados...")
        
        df = df.withColumn("_silver_ingestion_timestamp", F.current_timestamp())
        df = df.withColumn("_silver_processing_date", F.current_date())
        df = df.withColumn("_data_quality_flag", F.lit("VALIDATED"))
        
        return df
    
    @staticmethod
    def optimize_types(df: SparkDataFrame) -> SparkDataFrame:
        """Otimiza tipos de dados para reduzir tamanho"""
        logger.debug("Otimizando tipos de dados...")
        
        # Auto-conversão de tipos mais eficientes
        for field in df.schema.fields:
            col_name = field.name
            
            # Tentar converter strings numéricas para números
            if isinstance(field.dataType, StringType):
                # Sample para detectar padrão
                sample = df.select(col_name).filter(F.col(col_name).isNotNull()).limit(1000)
                # Aqui poderia adicionar lógica mais sofisticada
        
        return df

logger.info("✅ DataTransformer definido")

---
## 9. ⚡ Pipeline de Processamento Paralelo

In [ ]:
def process_table_to_silver(
    table_name: str,
    sample_size: int = None,
    sample_method: str = "RANDOM",
    remove_dups: bool = True,
    handle_nulls: bool = False,
    standardize: bool = True,
    run_quality_checks: bool = True
) -> PipelineMetrics:
    """
    Processa uma tabela Bronze -> Silver
    
    Args:
        table_name: Nome da tabela no ClickHouse
        sample_size: Número de linhas para amostrar (None = todas)
        sample_method: "RANDOM" ou "TOP"
        remove_dups: Remover duplicatas
        handle_nulls: Tratar valores nulos
        standardize: Padronizar strings
        run_quality_checks: Executar validações de qualidade
    
    Returns:
        PipelineMetrics com resultados do processamento
    """
    
    # Criar client separado para cada thread
    thread_client = clickhouse_connect.get_client(
        host=CH_HOST,
        port=CH_PORT,
        username=CH_USER,
        password=CH_PASSWORD,
        database=CH_DATABASE
    )
    
    logger.info(f"{'='*80}")
    logger.info(f"🔄 Processando: {table_name}")
    if sample_size:
        logger.info(f"🧪 Modo amostra: {sample_size:,} linhas ({sample_method})")
    logger.info(f"{'='*80}")
    
    # Inicializar métricas
    metrics = PipelineMetrics(
        table_name=table_name,
        start_time=datetime.now()
    )
    
    try:
        # 1. Ler dados do ClickHouse (Bronze) com amostragem
        logger.info(f"📥 Lendo dados de {table_name}...")
        
        if sample_size:
            if sample_method == "RANDOM":
                # Amostragem aleatória
                query = f"""
                    SELECT * 
                    FROM {table_name} 
                    ORDER BY rand() 
                    LIMIT {sample_size}
                """
            else:  # TOP
                # Primeiras N linhas
                query = f"""
                    SELECT * 
                    FROM {table_name} 
                    LIMIT {sample_size}
                """
        else:
            # Todas as linhas
            query = f"SELECT * FROM {table_name}"
        
        df_bronze = thread_client.query_df(query)
        
        if df_bronze.empty:
            logger.warning(f"⚠️  Tabela {table_name} está vazia")
            metrics.status = "skipped"
            metrics.finalize()
            thread_client.close()
            return metrics
        
        metrics.rows_input = len(df_bronze)
        logger.info(f"   Linhas lidas: {metrics.rows_input:,}")
        
        # 2. Converter para Spark DataFrame
        logger.info("⚡ Convertendo para Spark...")
        df_spark = spark.createDataFrame(df_bronze)
        
        # Cache para otimizar operações múltiplas
        df_spark.cache()
        
        initial_count = df_spark.count()
        logger.info(f"   Linhas no Spark: {initial_count:,}")
        
        # 3. Data Quality Checks (antes da transformação)
        if run_quality_checks:
            logger.info("🔍 Executando Data Quality checks...")
            qc = DataQualityChecker(df_spark, table_name)
            qc.check_completeness(threshold=0.70)  # 70% para amostra
            _, dup_count = qc.check_duplicates()
            metrics.rows_duplicates = dup_count
            
            summary = qc.get_summary()
            metrics.quality_checks_passed = summary['checks_passed']
            metrics.quality_checks_failed = summary['checks_failed']
            metrics.quality_score = summary['quality_score']
            
            # Salvar relatório de qualidade
            quality_report_path = QUALITY_DIR / f"{table_name}_quality_report.json"
            with open(quality_report_path, 'w') as f:
                json.dump(summary, f, indent=2)
        
        # 4. Aplicar transformações
        logger.info("🔧 Aplicando transformações...")
        transformer = DataTransformer()
        
        df_silver = df_spark
        
        if remove_dups:
            df_silver, removed = transformer.remove_duplicates(df_silver)
            metrics.rows_duplicates = removed
        
        if standardize:
            df_silver = transformer.standardize_strings(df_silver)
        
        if handle_nulls:
            # Para amostra, só remove linhas com muitos nulls
            df_silver = transformer.handle_nulls(df_silver, strategy='drop')
        
        # Adicionar metadados
        df_silver = transformer.add_metadata_columns(df_silver)
        
        # Adicionar flag de amostra
        if sample_size:
            df_silver = df_silver.withColumn("_is_sample", F.lit(True))
            df_silver = df_silver.withColumn("_sample_size", F.lit(sample_size))
        else:
            df_silver = df_silver.withColumn("_is_sample", F.lit(False))
        
        metrics.rows_output = df_silver.count()
        
        # 5. Escrever para ClickHouse (Silver)
        logger.info(f"💾 Gravando em {table_name}_silver...")
        
        # Converter para Pandas usando método alternativo (sem Arrow)
        # Coletar dados e criar DataFrame manualmente
        columns = df_silver.columns
        rows = df_silver.collect()
        data = [row.asDict() for row in rows]
        df_silver_pd = pd.DataFrame(data, columns=columns)
        
        # Criar tabela Silver no ClickHouse
        silver_table = f"{table_name}_silver"
        
        # Dropar se existir
        try:
            thread_client.command(f"DROP TABLE IF EXISTS {silver_table}")
            logger.debug(f"   Tabela {silver_table} dropada")
        except Exception as e:
            logger.debug(f"   Erro ao dropar tabela: {e}")
        
        # Inserir dados
        thread_client.insert_df(silver_table, df_silver_pd)
        
        logger.info(f"✅ {metrics.rows_output:,} linhas gravadas em {silver_table}")
        
        # Unpersist cache
        df_spark.unpersist()
        
        # 6. Finalizar métricas
        metrics.status = "success"
        metrics.finalize()
        
        logger.info(f"✅ Pipeline concluído com sucesso!")
        logger.info(f"   Input: {metrics.rows_input:,} linhas")
        logger.info(f"   Output: {metrics.rows_output:,} linhas")
        logger.info(f"   Duplicatas removidas: {metrics.rows_duplicates:,}")
        logger.info(f"   Tempo: {metrics.duration_seconds:.2f}s")
        logger.info(f"   Throughput: {metrics.throughput_rows_per_sec:.0f} rows/s")
        logger.info(f"   Quality Score: {metrics.quality_score:.1f}%")
        
        # Fechar client da thread
        thread_client.close()
        
        return metrics
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {table_name}: {e}")
        import traceback
        logger.error(traceback.format_exc())
        metrics.status = "failed"
        metrics.error_message = str(e)
        metrics.finalize()
        thread_client.close()
        return metrics

logger.info("✅ Pipeline de processamento (com amostragem) definido")

---
## 10. 🚀 Execução Paralela do Pipeline

In [ ]:
from typing import List
import threading

def process_tables_parallel(
    tables: List[str],
    max_workers: int = 2,  # Reduced from 4 to avoid overwhelming Spark
    **kwargs
) -> List[PipelineMetrics]:
    """
    Processa múltiplas tabelas em paralelo usando ThreadPoolExecutor
    
    Args:
        tables: Lista de nomes de tabelas
        max_workers: Número máximo de threads paralelas
        **kwargs: Argumentos para process_table_to_silver
    
    Returns:
        Lista de métricas de cada tabela processada
    """
    logger.info(f"🚀 Iniciando processamento paralelo de {len(tables)} tabelas")
    logger.info(f"   Max workers: {max_workers}")
    
    all_metrics = []
    start_time = datetime.now()
    
    # Process sequentially to avoid Spark JVM crashes
    for table in tables:
        try:
            logger.info(f"Processing table: {table}")
            metrics = process_table_to_silver(table, **kwargs)
            all_metrics.append(metrics)
            metrics_collector.add_metric(metrics)
            
            completed = len(all_metrics)
            pct = (completed / len(tables)) * 100
            logger.info(f"📊 Progresso: {completed}/{len(tables)} ({pct:.1f}%)")
            
        except Exception as e:
            logger.error(f"❌ Erro ao processar {table}: {e}")
            # Create failed metric
            failed_metric = PipelineMetrics(
                table_name=table,
                start_time=datetime.now(),
                status="failed",
                error_message=str(e)
            )
            failed_metric.finalize()
            all_metrics.append(failed_metric)
    
    total_duration = (datetime.now() - start_time).total_seconds()
    
    logger.info(f"✅ Processamento concluído!")
    logger.info(f"   Tempo total: {total_duration:.2f}s")
    logger.info(f"   Tabelas processadas: {len(all_metrics)}")
    
    return all_metrics

logger.info("✅ Função de processamento paralelo definida")

---
## 11. 🎬 Executar Pipeline

In [ ]:
# Configurar processamento
BATCH_SIZE = 10  # Processar em lotes
MAX_WORKERS = 1  # Sequential processing to avoid Spark crashes

logger.info(f"🎬 Iniciando pipeline Silver Layer")
logger.info(f"   Tabelas disponíveis: {len(available_tables)}")
logger.info(f"   Batch size: {BATCH_SIZE}")
logger.info(f"   Max workers: {MAX_WORKERS}")

# Processar em lotes para não sobrecarregar
all_results = []

for i in range(0, len(available_tables), BATCH_SIZE):
    batch = available_tables[i:i+BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1
    total_batches = (len(available_tables) + BATCH_SIZE - 1) // BATCH_SIZE
    
    logger.info(f"\n{'='*80}")
    logger.info(f"📦 Processando BATCH {batch_num}/{total_batches}")
    logger.info(f"   Tabelas: {', '.join(batch[:3])}...")
    logger.info(f"{'='*80}\n")
    
    batch_results = process_tables_parallel(
        tables=batch,
        max_workers=MAX_WORKERS,
        sample_size=SAMPLE_SIZE if SAMPLE_MODE else None,
        sample_method=SAMPLE_METHOD,
        remove_dups=True,
        standardize=True,
        run_quality_checks=True
    )
    
    all_results.extend(batch_results)
    
    # Pausa entre batches
    if i + BATCH_SIZE < len(available_tables):
        logger.info("⏸️  Pausa de 5s entre batches...")
        time.sleep(5)

logger.info(f"\n🎉 Pipeline completo!")
logger.info(f"   Total processado: {len(all_results)} tabelas")

---
## 12. 📊 Dashboard de Monitoramento e Métricas

In [ ]:
# Salvar métricas
metrics_file = metrics_collector.save_metrics()
logger.info(f"💾 Métricas salvas: {metrics_file}")

# Gerar sumário
summary = metrics_collector.get_summary()

print("\n" + "="*80)
print("📊 RESUMO GERAL DO PIPELINE")
print("="*80)
print(f"Tabelas processadas: {summary['total_tables_processed']}")
print(f"Linhas de entrada:   {summary['total_rows_input']:,}")
print(f"Linhas de saída:     {summary['total_rows_output']:,}")
print(f"Duplicatas removidas: {summary['total_duplicates_removed']:,}")
print(f"Linhas inválidas:    {summary['total_invalid_rows']:,}")
print(f"Quality Score médio: {summary['avg_quality_score']:.2f}%")
print(f"Taxa de sucesso:     {summary['success_rate']:.2f}%")
print("="*80)

In [ ]:
# Criar DataFrame de métricas para análise
metrics_df = pd.DataFrame([m.to_dict() for m in all_results])

# Visualizações
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Volumetria: Input vs Output',
        'Taxa de Duplicatas por Tabela',
        'Quality Score Distribuição',
        'Throughput (rows/sec)'
    ),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'histogram'}, {'type': 'bar'}]]
)

# 1. Volumetria
top_tables = metrics_df.nlargest(10, 'rows_input')
fig.add_trace(
    go.Bar(name='Input', x=top_tables['table_name'], y=top_tables['rows_input']),
    row=1, col=1
)
fig.add_trace(
    go.Bar(name='Output', x=top_tables['table_name'], y=top_tables['rows_output']),
    row=1, col=1
)

# 2. Taxa de duplicatas
metrics_df['dup_rate'] = (metrics_df['rows_duplicates'] / metrics_df['rows_input'] * 100).fillna(0)
top_dups = metrics_df.nlargest(10, 'dup_rate')
fig.add_trace(
    go.Bar(x=top_dups['table_name'], y=top_dups['dup_rate'], name='Dup %'),
    row=1, col=2
)

# 3. Quality Score
fig.add_trace(
    go.Histogram(x=metrics_df['quality_score'], name='Quality Score'),
    row=2, col=1
)

# 4. Throughput
top_throughput = metrics_df.nlargest(10, 'throughput_rows_per_sec')
fig.add_trace(
    go.Bar(x=top_throughput['table_name'], y=top_throughput['throughput_rows_per_sec'], name='Throughput'),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title_text="Silver Layer Pipeline - Dashboard de Métricas",
    showlegend=True
)

fig.update_xaxes(tickangle=45)

# Salvar dashboard
dashboard_path = METRICS_DIR / f"dashboard_{datetime.now():%Y%m%d_%H%M%S}.html"
fig.write_html(str(dashboard_path))

logger.info(f"📊 Dashboard salvo: {dashboard_path}")

fig.show()

---
## 13. 📈 Relatório de Data Quality

In [ ]:
# Analisar arquivos de quality reports
quality_reports = []

for report_file in QUALITY_DIR.glob("*_quality_report.json"):
    with open(report_file, 'r') as f:
        report = json.load(f)
        quality_reports.append(report)

if quality_reports:
    quality_df = pd.DataFrame(quality_reports)
    
    print("\n" + "="*80)
    print("✅ RELATÓRIO DE DATA QUALITY")
    print("="*80)
    print(f"\nTabelas analisadas: {len(quality_df)}")
    print(f"Quality Score médio: {quality_df['quality_score'].mean():.2f}%")
    print(f"\nTop 5 melhores tabelas:")
    print(quality_df.nlargest(5, 'quality_score')[['table_name', 'quality_score', 'checks_passed']])
    
    print(f"\nTabelas com issues:")
    issues_df = quality_df[quality_df['checks_failed'] > 0]
    if not issues_df.empty:
        print(issues_df[['table_name', 'checks_failed', 'quality_score']])
    else:
        print("✅ Nenhuma tabela com falhas!")
    
    print("="*80)
else:
    logger.warning("⚠️  Nenhum relatório de qualidade encontrado")

---
## 14. 🔍 Validação Final

In [ ]:
# Verificar tabelas Silver criadas
logger.info("🔍 Verificando tabelas Silver no ClickHouse...")

silver_tables = client.query_df("""
    SELECT 
        name,
        total_rows,
        total_bytes,
        formatReadableSize(total_bytes) as size
    FROM system.tables
    WHERE name LIKE '%_silver'
    ORDER BY total_rows DESC
""")

print("\n" + "="*80)
print("🥈 TABELAS SILVER CRIADAS")
print("="*80)
print(f"\nTotal: {len(silver_tables)} tabelas")

if len(silver_tables) > 0:
    print(f"\nTop 10 por volumetria:\n")
    print(silver_tables.head(10).to_string(index=False))
    print("="*80)
    
    # Estatísticas gerais
    total_rows = silver_tables['total_rows'].sum()
    total_size = silver_tables['total_bytes'].sum()
    
    print(f"\n📊 ESTATÍSTICAS GERAIS")
    print(f"Total de linhas: {total_rows:,}")
    print(f"Tamanho total: {total_size / (1024**3):.2f} GB")
    print(f"Média por tabela: {total_rows / len(silver_tables):,.0f} linhas")
else:
    print("\n⚠️  Nenhuma tabela Silver encontrada.")
    print("Verifique se o pipeline foi executado com sucesso.")
    print("="*80)

---
## 15. 🎓 Conclusões e Próximos Passos

### ✅ O que foi implementado:

1. **Arquitetura Medallion - Camada Silver**
   - Limpeza e padronização de dados
   - Remoção de duplicatas
   - Tratamento de valores nulos

2. **Data Quality Framework**
   - Validações automáticas (completude, duplicatas, tipos)
   - Quality score por tabela
   - Relatórios detalhados

3. **Observabilidade**
   - Logging estruturado (JSON)
   - Métricas de performance
   - Dashboard interativo

4. **Performance**
   - Processamento paralelo com ThreadPool
   - Otimizações Spark (Adaptive Query Execution)
   - Caching estratégico

### 🚀 Próximos Passos:

1. **Camada Gold**
   - Criar agregações e modelos dimensionais
   - Implementar SCD (Slowly Changing Dimensions)
   - Criar views otimizadas para BI

2. **Orquestração**
   - Integrar com Airflow/Prefect
   - Scheduling automático
   - Retry logic e alertas

3. **Monitoramento Avançado**
   - Integração com Prometheus/Grafana
   - Alertas de qualidade de dados
   - SLA tracking

4. **CI/CD**
   - Testes automatizados de data quality
   - Deploy automatizado
   - Rollback strategy

### 📚 Referências:

- [Medallion Architecture](https://www.databricks.com/glossary/medallion-architecture)
- [Great Expectations](https://docs.greatexpectations.io/)
- [Spark Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html)
- [ClickHouse Best Practices](https://clickhouse.com/docs/en/guides/best-practices/)